In [20]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd

# ==============================================================================
# PASSO 1: SIMULAÇÃO DE DADOS REAIS (ESTRUTURA WALMART M5)
# ==============================================================================
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):
    """
    Gera um dataset simulando a dinâmica da competição M5 da Walmart:
    - Vendas de contagem (Poisson/Binomial Negativa) com zeros (intermitência).
    - Variações de preço com elasticidade-preço.
    - Sazonalidade semanal e eventos especiais.
    """
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            # Demanda base do SKU na loja
            base_demand = np.random.uniform(0.5, 5.0)
            
            # Fator de sazonalidade semanal (fds vende mais)
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            # Variação e promoção de preço
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            # Dias com eventos/promoções especiais
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            # Parâmetro lambda da demanda diária
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            # Vendas reais observadas (intermitentes)
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                    'day_of_week': d_date.dayofweek, # 0=Segunda, 6=Domingo
                    'month': d_date.month,
                    'day_of_year': d_date.dayofyear
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365) # Função criada anteriormente

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [21]:
# 2. Configurar o MLForecast com as classes nativas da Nixtla
fcst = MLForecast(
    models={}, # Dicionário vazio caso não queira treinar modelos agora
    freq='D',
    lags=[7, 14, 28], # Lags automáticos por unique_id
    lag_transforms={
        # Aplica média e desvio padrão móveis no lag 1 com janela de 7 dias
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month'] # Features de calendário automáticas
)

df_prepared_nixtla = fcst.preprocess(
    df_nixtla, 
    id_col='unique_id', 
    time_col='ds', 
    target_col='y',
    static_features=[]
)

In [34]:
import pandas as pd
import numpy as np
import holidays
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd
import pandas as pd
import holidays

# 1. Definir os feriados dos anos desejados
us_holidays = holidays.US(years=range(2022, 2027))

# 2. Pré-calcular o conjunto de datas da janela FORA da função (uma única vez na memória)
_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1):  # Véspera (-2, -1) e o próprio dia (0)
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


# 3. Função limpa e ultra-rápida para o MLForecast
def is_holiday_window(dates) -> pd.Series:
    """Retorna 1 para o dia do feriado e os 2 dias que o antecedem."""
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

# Uso no MLForecast:
df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])

# 3. Integrar diretamente ao MLForecast
fcst = MLForecast(
    models={},
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    # Passamos funções personalizadas diretamente na lista date_features!
    date_features=['dayofweek', 'month']
)

# 4. Processar o DataFrame
# O Nixtla aplicará a função 'is_holiday' automaticamente durante o preprocess
df_prepared_nixtla = fcst.preprocess(
    df_nixtla, 
    id_col='unique_id', 
    time_col='ds', 
    target_col='y',
    static_features=[]
)

In [37]:
import pandas as pd
import numpy as np
import lightgbm as lgb

def estruturar_demanda_historica_issm(df_prepared):
    """
    Etapa 2: Extrai a taxa latente de demanda (lambda_t) limpa de ruído.
    
    Recebe o DataFrame pré-processado (com unique_id, ds, y, e exógenas como is_holiday_window)
    e treina um modelo com função de perda de Poisson para filtrar o ruído e extrair a média latente.
    """
    
    # 1. Separar variáveis explicativas estruturais (Sazonalidade, Eventos, Preço)
    features_estruturais = [
        'dayofweek', 'month', 'is_holiday_window', 'sell_price'
    ]
    
    # Adicionar lags e médias móveis se gerados no preprocess do MLForecast
    lag_cols = [c for c in df_prepared.columns if 'lag' in c or 'rolling' in c]
    features_totais = features_estruturais + lag_cols
    
    X = df_prepared[features_totais]
    y = df_prepared['y']
    
    # 2. Configurar o estimador com Objective 'poisson' ou 'tweedie'
    # Isso garante a estimativa do valor esperado latente lambda > 0 em dados com zero/intermitência
    model_issm = lgb.LGBMRegressor(
        objective='poisson',           # Adequado para contagem e intermitência
        metric='rmse',
        n_estimators=100,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    )
    
    # 3. Ajustar o modelo para separar Sinal vs Ruído
    model_issm.fit(X, y)
    
    # 4. Extrair a Taxa de Demanda Latente Estruturada (lambda_t)
    df_resultado = df_prepared.copy()
    df_resultado['lambda_t'] = model_issm.predict(X)
    
    # 5. Isolamento dos Componentes (Opcional - Para Inspecionar o ISSM)
    # Exemplo: Isolando o impacto puro de Feriados/Eventos
    X_sem_feriado = X.copy()
    X_sem_feriado['is_holiday_window'] = 0
    df_resultado['lambda_sem_feriado'] = model_issm.predict(X_sem_feriado)
    df_resultado['efeito_evento'] = df_resultado['lambda_t'] - df_resultado['lambda_sem_feriado']
    
    return df_resultado, model_issm

# --- Uso no Pipeline ---
# df_estruturado conterá a nova coluna 'lambda_t' pronta para o Bloco 3
df_estruturado, model_issm = estruturar_demanda_historica_issm(df_prepared_nixtla)

In [43]:
df_estruturado.loc[df_estruturado["efeito_evento"] > 0,['unique_id', 'ds', 'y', 'lambda_t', 'efeito_evento']]

,unique_id,ds,y,lambda_t,efeito_evento
182,STORE_01_FOODS_1_001,2023-07-02,3.0,1.895787,0.020108
311,STORE_01_FOODS_1_001,2023-11-08,3.0,1.483875,0.011444
364,STORE_01_FOODS_1_001,2023-12-31,2.0,2.183388,0.023159
414,STORE_01_FOODS_1_002,2023-02-19,6.0,2.900049,0.030760
511,STORE_01_FOODS_1_002,2023-05-27,5.0,2.419907,0.018663
609,STORE_01_FOODS_1_002,2023-09-02,5.0,4.656897,0.010152
645,STORE_01_FOODS_1_002,2023-10-08,5.0,2.971078,0.031514
722,STORE_01_FOODS_1_002,2023-12-24,2.0,2.712722,0.028773
778,STORE_01_FOODS_1_003,2023-02-18,3.0,2.239287,0.017270
877,STORE_01_FOODS_1_003,2023-05-28,2.0,1.950499,0.020689


In [38]:
print(df_estruturado[['unique_id', 'ds', 'y', 'lambda_t', 'efeito_evento']].head(10))

               unique_id         ds    y  lambda_t  efeito_evento
28  STORE_01_FOODS_1_001 2023-01-29  0.0  1.790130            0.0
29  STORE_01_FOODS_1_001 2023-01-30  3.0  2.045307            0.0
30  STORE_01_FOODS_1_001 2023-01-31  3.0  2.167702            0.0
31  STORE_01_FOODS_1_001 2023-02-01  2.0  1.267820            0.0
32  STORE_01_FOODS_1_001 2023-02-02  7.0  1.931008            0.0
33  STORE_01_FOODS_1_001 2023-02-03  4.0  2.986706            0.0
34  STORE_01_FOODS_1_001 2023-02-04  5.0  2.971639            0.0
35  STORE_01_FOODS_1_001 2023-02-05  1.0  2.360248            0.0
36  STORE_01_FOODS_1_001 2023-02-06  1.0  2.930025            0.0
37  STORE_01_FOODS_1_001 2023-02-07  3.0  2.846671            0.0


In [45]:
import numpy as np
import pandas as pd
from scipy.stats import nbinom
from scipy.optimize import minimize

# =====================================================================
# ETAPA 3: ESTIMATIVA DO PARÂMETRO DE DISPERSÃO (r)
# =====================================================================

def _nbinom_log_likelihood(params, y, lambda_t):
    """
    Função de Log-Verossimilhança Negativa para a Binomial Negativa
    Parametrização baseada em Média (lambda) e Dispersão (r)
    """
    r = params[0]
    if r <= 0:
        return 1e10
    
    # Mapeamento para os parâmetros padrão da scipy (p = r / (r + lambda))
    p = r / (r + lambda_t)
    
    # Calcular Log-Likelihood
    log_lik = nbinom.logpmf(y, r, p)
    return -np.sum(log_lik)

def estimar_parametro_dispersao(df_estruturado):
    """
    Estima o parâmetro r de sobredispersão por SKU/Loja.
    """
    df_resultado = df_estruturado.copy()
    
    # Dicionário para armazenar o valor de 'r' por série (unique_id)
    r_dict = {}
    
    for uid, group in df_resultado.groupby('unique_id'):
        y_obs = group['y'].values
        lambdas = group['lambda_t'].values
        
        # Chute inicial para r
        r_init = [1.0]
        
        # Otimização por Máxima Verossimilhança
        res = minimize(
            _nbinom_log_likelihood, 
            x0=r_init, 
            args=(y_obs, lambdas),
            method='L-BFGS-B',
            bounds=[(1e-4, 1e4)]
        )
        
        r_dict[uid] = res.x[0]
        
    df_resultado['r_dispersion'] = df_resultado['unique_id'].map(r_dict)
    return df_resultado

# =====================================================================
# ETAPA 4 e 6: GERAÇÃO DOS QUANTIS DA DISTRIBUIÇÃO PROBABILÍSTICA
# =====================================================================

def gerar_quantis_demanda(df_com_dispersao, quantis=[0.50, 0.67, 0.95, 0.99]):
    """
    Converte a distribuição Binomial Negativa N(lambda_t, r) 
    nos quantis numéricos de previsão para estoque.
    """
    df_quantis = df_com_dispersao.copy()
    
    for q in quantis:
        col_name = f'q_{int(q*100)}'
        
        # Parâmetro 'p' da distribuição no SciPy
        p_param = df_quantis['r_dispersion'] / (df_quantis['r_dispersion'] + df_quantis['lambda_t'])
        
        # Quantil discreto da Binomial Negativa (Percent Point Function - ppf)
        df_quantis[col_name] = nbinom.ppf(q, df_quantis['r_dispersion'], p_param)
        
    return df_quantis

# =====================================================================
# EXECUÇÃO DO PIPELINE (ETAPAS 3 A 6)
# =====================================================================

# 1. Estimar o parâmetro r de cada SKU/Loja
df_etapa3 = estimar_parametro_dispersao(df_estruturado)

# 2. Gerar as saídas probabilísticas em quantis (Etapas 4 e 6)
quantis_alvo = [0.50, 0.67, 0.95, 0.99]
df_saida_final = gerar_quantis_demanda(df_etapa3, quantis=quantis_alvo)

In [50]:
import numpy as np
import pandas as pd

def pinball_loss(y_true, y_pred, quantile):
    """
    Calcula a Pinball Loss para um quantil específico (tau).
    """
    err = y_true - y_pred
    return np.maximum(quantile * err, (quantile - 1) * err).mean()

def avaliar_quantis_pipeline(df_saida_final, quantis=[0.50, 0.67, 0.95, 0.99]):
    """
    Etapa 4: Avalia a qualidade dos quantis gerados pela Binomial Negativa.
    """
    resultados = {}
    
    for q in quantis:
        col_q = f'q_{int(q*100)}'
        
        # 1. Calculando a Pinball Loss do quantil
        loss = pinball_loss(
            y_true=df_saida_final['y'], 
            y_pred=df_saida_final[col_q], 
            quantile=q
        )
        
        # 2. Checando a Cobertura Real (Empirical Coverage)
        # Em q=0.95, a venda real 'y' DEVE ser menor ou igual ao quantil 'q_95' em cerca de 95% dos dias.
        cobertura_real = (df_saida_final['y'] <= df_saida_final[col_q]).mean()
        
        resultados[f'q_{int(q*100)}'] = {
            'Pinball_Loss': round(loss, 4),
            'Cobertura_Alvo': f"{int(q*100)}%",
            'Cobertura_Empirica': f"{cobertura_real*100:.1f}%"
        }
        
    return pd.DataFrame(resultados).T

# --- Execução no Pipeline ---
# Avaliando os resultados que vieram do df_saida_final da Etapa 3/6
df_avaliacao = avaliar_quantis_pipeline(df_saida_final)
print(df_avaliacao)

     Pinball_Loss Cobertura_Alvo Cobertura_Empirica
q_50       0.5855            50%              63.4%
q_67       0.5581            67%              78.4%
q_95       0.1813            95%              97.4%
q_99       0.0509            99%              99.6%
